# 24 - Final Official RAG UI Demo

Interactive demo for the selected official-law system. Recommended first run: retrieval-only. Then enable LLM generation for selected questions because Qwen3-32B is slow.

In [ ]:
!pip install -q -U gradio "sentence-transformers>=3.0.0" transformers accelerate bitsandbytes peft faiss-cpu rank-bm25 pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

OFFICIAL_INDEX = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'
BASE_LLM = 'Qwen/Qwen3-32B'
BASE_RERANKER = 'Qwen/Qwen3-Reranker-8B'
FINETUNED_LLM_ADAPTER = DRIVE_ROOT / 'models/adapters/qwen3_32b_qlora_combined_v1'

print('Device:', device)
print('Official Qwen3 embedding index exists:', OFFICIAL_INDEX.exists())
print('Fine-tuned LLM adapter exists:', FINETUNED_LLM_ADAPTER.exists())

In [ ]:
from functools import lru_cache
import gc
from typing import Any

import gradio as gr

from src.generation import generate_text, load_llm
from src.prompting import build_rag_prompt
from src.reranking import CrossEncoderReranker
from src.retrieval import RetrievalEngine

@lru_cache(maxsize=1)
def get_engine():
    return RetrievalEngine(index_root=OFFICIAL_INDEX, device=device)

def get_reranker(model_name: str):
    return CrossEncoderReranker(model_name=model_name, device=device)

@lru_cache(maxsize=2)
def get_llm(model_name: str, adapter_path_text: str, load_in_4bit: bool):
    adapter_path = Path(adapter_path_text) if adapter_path_text else None
    return load_llm(
        model_name=model_name,
        device=device,
        load_in_4bit=load_in_4bit,
        adapter_path=adapter_path if adapter_path and adapter_path.exists() else None,
    )

def format_sources(items: list[dict[str, Any]]) -> str:
    lines = []
    for i, item in enumerate(items, start=1):
        citation = item.get('citation_label') or item.get('article_key') or ''
        article_key = item.get('article_key', '')
        score = item.get('score', '')
        text = str(item.get('generation_text') or item.get('retrieval_text') or '')
        text = text.replace('\n', ' ').strip()[:850]
        lines.append(f'[{i}] {citation}\narticle_key={article_key} | score={score}\n{text}')
    return '\n\n'.join(lines)

def retrieve_context(question: str, use_reranker: bool, candidate_k: int, top_k_context: int, reranker_batch_size: int):
    engine = get_engine()
    if use_reranker:
        candidates = engine.dense_search(question, top_k=candidate_k)
        reranker = get_reranker(BASE_RERANKER)
        reranked = reranker.rerank(
            query=question,
            candidates=candidates,
            text_field='retrieval_text',
            top_k=top_k_context,
            batch_size=reranker_batch_size,
        )
        del reranker
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()
        return reranked
    return engine.dense_search(question, top_k=top_k_context)

def answer_question(
    question: str,
    generate_answer: bool,
    use_reranker: bool,
    llm_variant: str,
    candidate_k: int,
    top_k_context: int,
    max_context_chars: int,
    max_new_tokens: int,
    reranker_batch_size: int,
):
    question = (question or '').strip()
    if not question:
        return 'Soru yaz.', ''
    if not OFFICIAL_INDEX.exists():
        return f'Index bulunamadı: {OFFICIAL_INDEX}', ''

    retrieved = retrieve_context(
        question=question,
        use_reranker=use_reranker,
        candidate_k=int(candidate_k),
        top_k_context=int(top_k_context),
        reranker_batch_size=int(reranker_batch_size),
    )
    sources = format_sources(retrieved)
    if not generate_answer:
        mode = 'Qwen3-Embedding-8B + Qwen3-Reranker-8B' if use_reranker else 'Qwen3-Embedding-8B dense'
        return f'Retrieval-only mode: {mode}\n\n{sources}', sources

    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    adapter_text = str(FINETUNED_LLM_ADAPTER) if llm_variant == 'Qwen3-32B QLoRA fine-tuned' else ''
    try:
        tokenizer, model = get_llm(BASE_LLM, adapter_text, True)
    except Exception as exc:
        return f'LLM yuklenemedi. Colab VRAM dolmus olabilir. Runtime restart edip sadece bu notebooku calistir veya once reranker checkboxini kapat. Hata: {type(exc).__name__}: {exc}', sources
    prompt = build_rag_prompt(question, retrieved, max_context_chars=int(max_context_chars))
    answer = generate_text(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_tokens=int(max_new_tokens),
        temperature=0.0,
        top_p=1.0,
        input_max_length=8192,
    )
    return answer, sources

with gr.Blocks(title='Turkish Legal RAG - Final System') as demo:
    gr.Markdown('## Turkish Legal RAG - Final Official System')
    gr.Markdown('Default retrieval is the strongest measured setup: Qwen3-Embedding-8B top-30 + Qwen3-Reranker-8B top-10. Use retrieval-only first for fast demos.')
    with gr.Row():
        question = gr.Textbox(label='Soru', lines=4, placeholder='Örn. Anayasa Mahkemesine bireysel başvuru şartları nelerdir?')
    with gr.Row():
        generate_answer = gr.Checkbox(value=False, label='LLM cevabı üret')
        use_reranker = gr.Checkbox(value=True, label='Qwen3-Reranker-8B kullan')
        llm_variant = gr.Dropdown(
            choices=['Qwen3-32B base', 'Qwen3-32B QLoRA fine-tuned'],
            value='Qwen3-32B base',
            label='LLM varyantı',
        )
    with gr.Row():
        candidate_k = gr.Slider(10, 50, value=30, step=5, label='Candidate K')
        top_k_context = gr.Slider(3, 15, value=10, step=1, label='Context Top K')
        reranker_batch_size = gr.Slider(1, 8, value=4, step=1, label='Reranker batch size')
    with gr.Row():
        max_context_chars = gr.Slider(3000, 16000, value=9000, step=1000, label='Max context chars')
        max_new_tokens = gr.Slider(128, 768, value=384, step=64, label='Max answer tokens')
    run_btn = gr.Button('Çalıştır')
    answer_box = gr.Textbox(label='Cevap / Retrieval sonucu', lines=14)
    sources_box = gr.Textbox(label='Kaynaklar', lines=14)

    run_btn.click(
        answer_question,
        inputs=[question, generate_answer, use_reranker, llm_variant, candidate_k, top_k_context, max_context_chars, max_new_tokens, reranker_batch_size],
        outputs=[answer_box, sources_box],
    )

demo.launch(share=True, debug=True)